# Visibility-Aware Visual Domain Erasure for Generalizable Person Re-Identification (DG-ReID)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kapish19/BTP/blob/main/DG_ReID_Colab.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-kapish19%2FBTP-blue.svg)](https://github.com/kapish19/BTP)

**B.Tech Project-I (BTP-I) Official Research Notebook**  
**Authors:** Diya Bangera (2023UCA1917), Kapish Verma (2023UCS1632), Rishit Rana (2023UCA1911)  
**Supervisor:** Dr. Vandana Bhatia, NSUT, New Delhi  

---

### Table of Contents
1. [Hardware & Environment Setup](#setup)
2. [Clone Repository & Install Dependencies](#clone)
3. [Automated Unit & Pipeline Tests](#tests)
4. [Instant Multi-Stage Smoke Test (Synthetic Data)](#smoke)
5. [Dataset Download & Directory Setup](#datasets)
6. [Full 3-Stage Training Pipeline](#train)
7. [Zero-Shot Cross-Domain & Occlusion Evaluation](#eval)
8. [Component-Wise Ablation Study (Table 7 & Table 8)](#ablations)
9. [Inference on Custom Pedestrian Images](#inference)

## 1. Hardware & Environment Setup <a name="setup"></a>
Ensure that GPU acceleration is enabled: `Runtime` -> `Change runtime type` -> select `T4 GPU` (or `A100` / `V100`).

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:      {torch.cuda.get_device_name(0)}")
    print(f"Device Memory:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 2. Clone Repository & Install Dependencies <a name="clone"></a>

In [ ]:
import os

# Clone the repository if not already cloned
if not os.path.exists('BTP'):
    !git clone https://github.com/kapish19/BTP.git

%cd BTP

# Install dependencies
!pip install -q -r requirements.txt
!pip install -q -e .

## 3. Automated Unit & Pipeline Tests <a name="tests"></a>
Runs all 22 tests verifying:
- CLIP ViT-B/16 shapes & bicubic positional embedding interpolation
- GRL forward identity, scheduled alpha, reversed gradient sign
- PartVisibilityGAT 3-node graph attention & visibility head
- Synthetic occlusion masks & soft visibility targets
- All 7 loss functions & gradient propagation
- Parameter trainability across Stages 1, 2, and 3
- CMC & mAP evaluation with camera exclusion rules
- Checkpoint roundtrip save/load numerical tolerance
- End-to-end multi-stage pipeline smoke test

In [ ]:
!python -m unittest discover tests -v

## 4. Instant Multi-Stage Smoke Test (Synthetic Data) <a name="smoke"></a>
Executes 3 complete epochs (Stage 1, Stage 2, Stage 3) on automatically generated multi-domain pedestrian crops.  
**No external dataset downloads needed!**

In [ ]:
# Run a 3-epoch multi-stage training test
!python train.py --config configs/default.yaml --synthetic --epochs 3 --output_dir ./outputs/synthetic_demo

In [ ]:
# Evaluate model across Clean, Mild (20%), Moderate (35%), and Severe (55%) occlusion
!python evaluate.py --config configs/default.yaml --synthetic --checkpoint ./outputs/synthetic_demo/checkpoints/checkpoint_best.pth --output_dir ./outputs/synthetic_eval

## 5. Dataset Download & Directory Setup <a name="datasets"></a>

The codebase supports automatic directory resolution for the following benchmark layouts:
- **Market-1501:** `Market-1501-v15.09.15/` ([Kaggle](https://www.kaggle.com/datasets/pengcw1/market-1501/data))
- **MSMT17:** `MSMT17_V1/` ([Kaggle](https://www.kaggle.com/datasets/ouassimaazzouzi/msmt17))
- **CUHK-SYSU:** `cuhk_sysu/` ([Kaggle](https://www.kaggle.com/datasets/manaschaiaonon/cuhk-sysu))
- **CUHK03:** `cuhk03/` ([Kaggle](https://www.kaggle.com/datasets/priyanagda/cuhk03) + [CUHK03-NP](https://github.com/zhunzhong07/person-re-ranking/tree/master/CUHK03-NP))
- **Occluded-DukeMTMC:** `Occluded-DukeMTMC/` ([GitHub](https://github.com/lightas/Occluded-DukeMTMC-Dataset))

### Option A: Download via Kaggle API (Recommended in Colab)
Uncomment and run below to download datasets directly into `./data/`:

In [ ]:
# Upload kaggle.json to download all benchmarks directly in Colab
# from google.colab import files
# files.upload()  # Upload your kaggle.json

# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

# Option 1: All-in-one automated downloader & verifier for all datasets
# !bash scripts/setup_datasets.sh

# Option 2: Or via Python CLI
# !python src/data/prepare_datasets.py --download all --dir ./data


### Option B: Mount Google Drive (if datasets are stored in Drive)

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# # Link to your Drive datasets folder:
# !ln -s /content/drive/MyDrive/reid_datasets ./data

### Verify Dataset Directories

In [ ]:
!python src/data/prepare_datasets.py --info

## 6. Full 3-Stage Training Pipeline <a name="train"></a>

Training protocol following Table 3 of the report:
- **Stage 1 (Epochs 1–10):** Visual backbone frozen; train projection heads, BNNecks, GAT, visibility head, domain factors.
- **Stage 2 (Epochs 11–30):** Unfreeze final 2 ViT transformer blocks (blocks 10 & 11).
- **Stage 3 (Epochs 31–60):** Activate domain classifier behind Gradient Reversal Layer with scheduled warmup $\alpha(e) = \min(1, \frac{e-30}{10})$.

In [ ]:
# Example: Train Leave-Market-1501-out Cross-Domain Model
# (Sources: MSMT17 + CUHK03 + CUHK-SYSU -> Target: Market-1501)

# Stage 1
!python train.py --config configs/market1501_dg.yaml --stage 1 --output_dir ./outputs/market_dg

# Stage 2
!python train.py --config configs/market1501_dg.yaml --stage 2 --resume ./outputs/market_dg/checkpoints/stage_1_final.pth --output_dir ./outputs/market_dg

# Stage 3
!python train.py --config configs/market1501_dg.yaml --stage 3 --resume ./outputs/market_dg/checkpoints/stage_2_final.pth --output_dir ./outputs/market_dg

## 7. Zero-Shot Cross-Domain & Occlusion Evaluation <a name="eval"></a>
Evaluates the best checkpoint on the unseen target domain under Clean (0%), Mild (20%), Moderate (35%), and Severe (55%) occlusion conditions.

In [ ]:
!python evaluate.py --config configs/market1501_dg.yaml --checkpoint ./outputs/market_dg/checkpoints/checkpoint_best.pth --output_dir ./outputs/market_dg/eval

In [ ]:
# Display the degradation plot inline
from IPython.display import Image, display
plot_file = './outputs/market_dg/eval/occlusion_degradation.png'
if os.path.exists(plot_file):
    display(Image(plot_file))

## 8. Component-Wise Ablation Study (Table 7 & Table 8) <a name="ablations"></a>

In [ ]:
# Display the official BTP Report Ground-Truth Tables (Markdown, ASCII, and LaTeX)
!python ablate.py --mode display

In [ ]:
# Run live ablation training on all 4 configurations:
# 1. Baseline CLIP Backbone
# 2. GRL-Only
# 3. Part-Only
# 4. Full DG-ReID
# !bash scripts/run_ablations.sh

## 9. Inference on Custom Pedestrian Images <a name="inference"></a>

In [ ]:
# Extract 512-dimensional normalized feature embedding for a pedestrian crop
!python inference.py --query ./outputs/synthetic_eval/synthetic_dataset/domain_2/query/0024_c1s1_000001_00.jpg

In [ ]:
# Rank gallery images against query crop
!python inference.py \
    --query ./outputs/synthetic_eval/synthetic_dataset/domain_2/query/0024_c1s1_000001_00.jpg \
    --gallery ./outputs/synthetic_eval/synthetic_dataset/domain_2/bounding_box_test \
    --top_k 5